# Задача: Прогнозиране на заплата

## Въведение:
Вие сте младши data scientist в компания и получавате задача да подобрите съществуващ модел за прогнозиране на заплати. Настоящият модел е проста линейна регресия, която използва само няколко основни признака от данните, събрани от Glassdoor.
Вашата цел е да използвате уменията си в областта на **feature engineering** и **моделирането**, за да създадете значително по-точен модел.

## 1. Проблемът

Ще ви бъде предоставен Python скрипт, който:

1. Зарежда и извършва минимално почистване на данните.
2. Обучава базов модел (**Линейна регресия**) върху малък набор от характеристики.
3. Изчислява и отпечатва неговата средна абсолютна грешка (Mean Absolute Error - MAE).

Вашата задача е да работите в обозначената "Зона за състезатели", за да:

- Създадете нови, по-информативни характеристики от съществуващите данни.
- Изберете и обучите по-мощен модел.
- Постигнете по-нисък MAE от базовия модел.

## 2. Оценка

Основният показател за оценка е Средна абсолютна грешка (MAE). По-ниска стойност означава по-добър модел - вашата цел е да я минимизирате. MAE е лесна за интерпретиране, тъй като показва средната грешка в прогнозата в същите единици като заплатата (напр. MAE от 15 означава, че моделът греши средно с $15,000).

## 3. Предаване

Предайте генерираните си прогнози predictions.csv върху данните от test_features.csv, заедно с тетрадката.



Импортиране на необходимите библиотеки

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import warnings
from sklearn.model_selection import GridSearchCV, cross_val_score

warnings.filterwarnings('ignore')

In [ ]:
BASELINE_FEATURES = ['Rating', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue']
Y_COLUMN = ['Salary Estimate']

In [ ]:
!pip install gdown
!gdown 1atCcqIxgKfvmtRUQkqz--wF2FTlEAHal
!gdown 1fA6iChCXW2e6v9a3G68r6b6L36_Em1ki

# Помощни функции и базов модел

In [34]:
# ==============================================================================
# === (ПОМОЩНИ ФУНКЦИИ И БАЗОВ МОДЕЛ - НЕ ПРОМЕНЯЙТЕ ТАЗИ КЛЕТКА) ===
# ==============================================================================

def title_simplifier(title):
    if 'data scientist' in title.lower():
        return 'data scientist'
    elif 'data engineer' in title.lower():
        return 'data engineer'
    elif 'analyst' in title.lower():
        return 'analyst'
    elif 'machine learning' in title.lower():
        return 'mle'
    elif 'manager' in title.lower():
        return 'manager'
    elif 'director' in title.lower():
        return 'director'
    else:
        return 'na'

def seniority(title):
    if 'sr' in title.lower() or 'senior' in title.lower() or 'lead' in title.lower() or 'principal' in title.lower():
        return 'senior'
    elif 'jr' in title.lower() or 'jr.' in title.lower():
        return 'jr'
    else:
        return 'na'

def load_data(path: str) -> pd.DataFrame:
    """
    Зарежда данните и извършва стабилно първоначално почистване,
    вдъхновено от подробния анализ.
    """
    df = pd.read_csv(path)
    return df

def clean_train_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df['Salary Estimate']!= '-1']

    # Почистване на стринга със заплатата
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.split('(')[0])
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.replace('K','').replace('$',''))

    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.lower().replace('per hour', ''))
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.lower().replace('employer provided salary:', ''))

    df['Rating'] = df['Rating'].apply(lambda x: x if x > 0 else np.nan)
    df['State'] = df.Location.apply(lambda x: x.split(',')[1])

    # Справяне както с диапазони (напр. "50-100"), така и с единични стойности ("75")
    df['Min_Salary'] = df['Salary Estimate'].apply(lambda x: int(x.split('-')[0]))
    df['Max_Salary'] = df['Salary Estimate'].apply(lambda x: int(x.split('-')[1]))
    df['Salary Estimate']= (df['Min_Salary'] + df['Max_Salary'])/2
    df.drop(['Min_Salary', 'Max_Salary'], axis=1, inplace=True)
    return df

def clean_data_features(df: pd.DataFrame) -> pd.DataFrame:
    df['Rating'] = df['Rating'].apply(lambda x: x if x > 0 else np.nan)
    df['State'] = df.Location.apply(lambda x: x.split(',')[1])
    return df

def train_baseline_model(df: pd.DataFrame) -> float:
    """Обучава базов модел и го връща, заедно с неговия MAE и ."""

    # Използваме няколко категорийни и числови характеристики
    base_features = BASELINE_FEATURES + Y_COLUMN
    df_base = df[base_features].dropna()

    # Създаване на dummy променливи за базовия модел
    df_base_dum = pd.get_dummies(df_base)
    X = df_base_dum.drop('Salary Estimate', axis=1)
    y = df_base_dum['Salary Estimate'].values

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    print(f"Baseline Model MAE: {mae:.3f}")
    return model, mae, X, y

# Зона за състезатели

## Цел:

Вашата задача е да имплементирате функцията student_solution по-долу. Целта е да създадете модел, който има по-ниска средна абсолютна грешка (MAE) от базовия модел.

## Стъпки:

1. Feature Engineering: Създайте нови, по-полезни характеристики.
2. Подготовка на данните: Изберете характеристиките, които ще използвате, и ги подгответе за модела по подходящ начин.
3. Обучение на модел: Изберете и обучете своя модел.

In [35]:
# ==============================================================================
# === (ГЛАВЕН СКРИПТ - НЕ ПРОМЕНЯЙТЕ ТАЗИ КЛЕТКА) ===
# ==============================================================================
# Зареждане на данните
df_1 = load_data("train.csv")
df = clean_train_data(df_1)

# Обучение и оценка на базовия модел
baseline_model, baseline_mae, X_train_baseline, y_train_baseline = train_baseline_model(df.copy())

Baseline Model MAE: 28.415


In [36]:
df_test = load_data("../../../test_features.csv")
df_test_cleaned = clean_data_features(df_test)
df_test_processed = df_test_cleaned[BASELINE_FEATURES].dropna()
baseline_model_columns = X_train_baseline.columns
# Създаване на dummy променливи за базовия модел
df_test_dum = pd.get_dummies(df_test_processed)
df_test_aligned = df_test_dum.reindex(
    columns=baseline_model_columns, fill_value=0
)
y_pred = baseline_model.predict(df_test_aligned)
y_pred_df = pd.DataFrame(y_pred, columns=Y_COLUMN)
# y_pred_df.to_csv('predictions.csv', index=False)

In [39]:
# ==============================================================================
# ЗОНА ЗА ВАШИЯ КОД
# =============================================================================
# Може да опитате да:
# - почистите/преработите допълнително данните (напр. )
# - добавите други характеристики (напр. възраст на компанията)
# - експериментирате с различни модели
# - комбинирате няколко модела
# - пуснете grid search за намиране на оптимални параметри
# ==============================================================================
student_model = baseline_model

In [ ]:
df[['Age','Salary Estimate', 'Rating', 'desc_len']].corr()

In [44]:
_, X_val, _, y_val = train_test_split(X_train_baseline, y_train_baseline, test_size=0.2, random_state=42)
student_predictions = student_model.predict(X_val)
student_mae = mean_absolute_error(y_val, student_predictions)

print("\n--- Финални резултати ---")
print(f"Базовият модел е с MAE: {baseline_mae:.3f}")
print(f"Твоят модел е с MAE:    {student_mae:.3f}")

improvement = baseline_mae - student_mae
if improvement > 0:
    print(f"\nПодобрение спрямо baseline модела: {improvement:.3f}!")
else:
    print("\nОпитай отново! Моделът ти не е по-добър от базовия.")


--- Финални резултати ---
Базовият модел е с MAE: 28.415
Твоят модел е с MAE:    28.415

Опитай отново! Моделът ти не е по-добър от базовия.
